# Image → 3D on Colab (Stable Fast 3D, free T4)

Turns a **single image** into a clean, UV-unwrapped, **textured** 3D mesh using
[Stable Fast 3D (SF3D)](https://github.com/Stability-AI/stable-fast-3d) on a free Colab T4 —
**no HuggingFace ZeroGPU quota limits.**

Why SF3D for Roblox UGC: it's a *direct mesh* model (not gaussian-splatting), so it emits
**one coherent mesh** instead of the fragmented splat geometry + attached backdrop planes that
TRELLIS produces. It exposes exactly the marketplace-prep controls we want:
**triangle/quad remesh**, a **vertex-count cap** (land near the Roblox tri budget), and a
**2048 texture** (the Roblox cap).

### How to run
1. **Runtime → Change runtime type → T4 GPU**, then **Save**.
2. Accept the model license once (cell 3 explains).
3. **Runtime → Run all**, upload your image when prompted, and a `.glb` downloads at the end.

First run takes ~3–5 min (install + first-time CUDA op build). After that each generation is seconds.

## 1. Check the GPU (must say Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Accept the license + log in to HuggingFace

SF3D's weights are **gated**. One-time steps:
1. Open <https://huggingface.co/stabilityai/stable-fast-3d> and click **Agree / Access repository**.
2. Create a **read** token at <https://huggingface.co/settings/tokens>.
3. Run the cell below and paste the token (it is not saved to the notebook).

In [ ]:
from huggingface_hub import login
import getpass
login(token=getpass.getpass('HF token (read scope): '))
print('logged in')

## 3. Install SF3D

Clones the official repo and installs deps. The custom CUDA ops (`texture_baker`,
`uv_unwrapper`) compile here with ninja — this is the slow part the first time.

In [ ]:
%cd /content
![ -d stable-fast-3d ] || git clone https://github.com/Stability-AI/stable-fast-3d.git
%cd /content/stable-fast-3d
# Colab's torch needs setuptools<82 — do NOT upgrade past it. Quote 'setuptools<82'
# so the shell doesn't read '<82' as a redirection.
!pip install -q 'setuptools<82' wheel ninja
!pip install -q -r requirements.txt
# The two CUDA ops import torch in their setup.py, so pip's ISOLATED build env
# (which has no torch) fails with "Getting requirements to build wheel ... No
# available output". --no-build-isolation makes the build reuse the installed torch.
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda'
!CUDA_HOME=/usr/local/cuda pip install -q --no-build-isolation ./texture_baker ./uv_unwrapper
import importlib
for _ext in ('texture_baker', 'uv_unwrapper'):
    importlib.import_module(_ext)  # raises loudly here if the build truly failed
print('SF3D + CUDA ops installed OK')

## 4. Upload your image

A clean subject on a plain background works best (SF3D removes the background automatically).
Run the cell, pick your file. To use a path instead, set `IMAGE_PATH` directly.

In [ ]:
from google.colab import files
import shutil, os
up = files.upload()
IMAGE_PATH = '/content/input' + os.path.splitext(next(iter(up)))[1]
shutil.move(next(iter(up)), IMAGE_PATH)
print('using', IMAGE_PATH)

## 5. Generate the mesh

Tunables:
- `REMESH` = `triangle` (clean even triangles, Roblox-friendly) / `quad` / `none`.
- `VERTEX_COUNT` = `-1` (auto) or a cap, e.g. `2000` to land near the rigid-accessory budget (4,000 tris).
- `TEXTURE_RES` = `2048` (Roblox cap).

In [ ]:
REMESH = 'triangle'
VERTEX_COUNT = -1
TEXTURE_RES = 2048

%cd /content/stable-fast-3d
!python run.py "$IMAGE_PATH" \
    --output-dir /content/output \
    --texture-resolution {TEXTURE_RES} \
    --remesh_option {REMESH} \
    --target_vertex_count {VERTEX_COUNT}

import glob
GLB = sorted(glob.glob('/content/output/**/*.glb', recursive=True))[-1]
print('mesh:', GLB)

## 6. Quick mesh stats + download

In [ ]:
!pip install -q trimesh
import trimesh
m = trimesh.load(GLB, force='mesh')
print(f'tris: {len(m.faces):,}   verts: {len(m.vertices):,}   watertight: {m.is_watertight}')
print(f'bounds (units): {m.extents.round(3)}')
from google.colab import files
files.download(GLB)

## 7. Back in the repo

Drop the downloaded `.glb` into `runs/` and continue the pipeline:

```bash
# optional safety net (SF3D is already clean, so this should be a no-op):
roblox-ugc clean runs/shark/mesh.glb --out runs/shark/clean.glb

# import to Blender, decimate to the category tri budget, center, rescale:
roblox-ugc prep runs/shark/clean.glb --out runs/shark/prepped.fbx --decimate 4000 --center
roblox-ugc inspect runs/shark/prepped.fbx --out runs/shark/report.json
roblox-ugc validate runs/shark/report.json --target accessory --category Hat
```

### Higher fidelity?
If you want richer geometry/texture and don't mind a slower run, **Hunyuan3D-2** also fits a T4
(<https://github.com/Tencent-Hunyuan/Hunyuan3D-2>) — octree mesh, also clean (no splat artifacts),
with controllable polygon count. SF3D is the fast/clean default; Hunyuan3D-2 is the quality step-up.

> Note: a **TPU** (e.g. v5e) can't run these models — they use custom CUDA kernels. Use the **T4 GPU** runtime.